# 2. Química Avanzada (Zonas Dinámicas)

`PA3Py` fue diseñado para ser completamente agnóstico en términos de composición. Tú puedes inyectar tus propias funciones químicas de Python y él automáticamente descubrirá las especies y construirá las tablas.

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('../../src'))
from pa3py import PA3Py

# Cargar simulación
sim = PA3Py('../../tests/test_data/run_smooth_a0.001_v10')

[load_tripodpy_hdf5] Leyendo 100 snapshots desde ../../tests/test_data/run_smooth_a0.001_v10...


## Función de Composición Personalizada

Definiremos un disco protoplanetario con 4 especies químicas (`silicatos`, `H2O`, `CO2`, `CO`) particionado en 4 zonas dinámicas.

In [2]:
from pa3py import constants as c

def quimica_4_zonas(r_cm, t_sec):
    # 1. Definimos las snowlines (algunas pueden migrar)
    r_h2o = 2.73 * c.AU * (max(t_sec, 1e-6) / 1e13)**(-0.5)
    r_co2 = 5.0 * c.AU
    r_co  = 12.0 * c.AU
    
    # 2. Asignamos abundancias relativas según la distancia al sol
    if r_cm < r_h2o:
        return {'silicatos': 1.0}                           # 100% rocoso
    elif r_cm < r_co2:
        return {'silicatos': 0.5, 'H2O': 0.5}               # Zona de Hielo
    elif r_cm < r_co:
        return {'silicatos': 0.3, 'H2O': 0.3, 'CO2': 0.4}   # Zona CO2
    else:
        return {'silicatos': 0.2, 'H2O': 0.2, 'CO2': 0.3, 'CO': 0.3} # Zona fría

# Inyectamos nuestra función al motor. PA3Py auto-detectará las especies.
sim.set_custom_chemistry(quimica_4_zonas)

## Corriendo la Química Personalizada

Colocaremos embriones en varias regiones y dejaremos que PA3Py se encargue del resto.

In [3]:
# Embriones en la zona rocosa, hielo, CO2 y CO
resultados = sim.run_growth([2.0, 4.0, 8.0, 15.0])


---------------------------------------------------------------------------------
  r [AU]  M_tot [ME]  M_iso [ME]  f_silicatos[%]  f_H2O[%]  f_CO2[%]   f_CO[%]
---------------------------------------------------------------------------------
    2.00        0.164         6.65            87.7      12.3       0.0       0.0
    4.00        0.306        11.19            63.8      36.2       0.0       0.0
    8.00        0.123        18.82            39.7      25.8      34.4       0.0
   15.00        0.001        30.15            82.7       4.3       6.5       6.5
---------------------------------------------------------------------------------

